# CS781 — pretrain a ~300M "B" model (your arch3: GQA + RoPE decoder) from scratch on AWS (g6e.2xlarge, 1x L40S)

**Before you start (instance):** Deep Learning AMI with PyTorch (GPU), **EBS volume >= 150 GB** (raw corpus ~20 GB +
token parts ~9 GB + train.bin ~9 GB + checkpoints ~3.6 GB each for last.pt), Jupyter on the instance.
Put **`legal_bpe.json`** (your Phase-1 tokenizer, sha256 38f51f14, from `~/capstone/phase1/ab_inputs/`) next to this
notebook, and export your Hugging Face token in the shell that starts Jupyter: `export HF_TOKEN=...` (never paste it here).

**Your parts (course rule 9):** every `None` in the configuration cell: architecture dims (target 250–300M params, the
Phase-1 limit is 300M), and all training hyperparameters (lr schedule, batch, token budget, init).
Everything else is plumbing (declare it): data build, training loop, probe, monitor, export.

**Course rule 7 reminder:** "Phases 2–7 must start from your Phase 1 (or later-phase) model." Confirm with the TA that
this from-scratch model may be used in Phase 3+ before relying on it.

Order: 0 checks → 1 config → 2 code files → 3 param count → 4 data (once, ~1–2 h CPU) → 5 probe (~10 min) →
6 launch in background (survives closing the browser) → 7 monitor → 8 export.

## 0 — environment

In [ ]:
# environment check
import os, sys, json, time, glob, shutil, hashlib, subprocess, math
import torch
WORK = os.path.abspath(os.getcwd())
if WORK not in sys.path:
    sys.path.insert(0, WORK)                  # the code files are written next to this notebook
print("python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA", torch.version.cuda)
assert torch.cuda.is_available(), "no GPU visible"
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name} {p.total_memory / 1e9:.0f} GB | bf16 {torch.cuda.is_bf16_supported()}")
print(f"CPUs {os.cpu_count()} | free disk here {shutil.disk_usage(WORK).free / 1e9:.0f} GB")
assert shutil.disk_usage(WORK).free > 60e9, "less than 60 GB free: enlarge the EBS volume"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "numpy", "matplotlib"], check=True)
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

## 1 — configuration (yours)

In [ ]:
# =====================================================================================
# YOUR CHOICES (course rule 9). Fill every None. Phase-1 limit: <= 300M parameters (target 250-300M).
# For reference, B2 was: d_model 768, n_layers 18, n_heads 12, n_kv_heads 4, d_ff 3456, max_seq_len 896, rope,
# norm_first True, attention_scale 0.125 (148.6M params); lr 3e-4 -> 3e-5 cosine, warmup 1000, init_std 0.02.
# =====================================================================================
TOKENIZER = os.path.join(WORK, "legal_bpe.json")
DATA_DIR = os.path.join(WORK, "data")
OUT_DIR = os.path.join(WORK, "run_B3")
TAG = "B3"

# YOUR B3 configuration (your values). 22 layers = 298,273,792 params (checked with your arch3 code; 21-layer fallback not needed).
MODEL = dict(
    vocab_size=32000,                 # fixed: must match legal_bpe.json (the Phase-2+ pipeline depends on it)
    d_model=1024, n_layers=22, n_heads=16, n_kv_heads=4, d_ff=4608,     # head dim 64, GQA 16 -> 4
    max_seq_len=896, dropout=0.1,
    pos_emb_type="rope", norm_first=True, attention_scale=0.125,       # 0.125 = 1/sqrt(64)
)
INIT_STD = 0.02                       # tied embedding / LM head (arch3 ties them)
TRAIN = dict(
    batch_tokens=16_128,              # your 16,384 is not a multiple of 896; 18 x 896 = 16,128 is what B2's "16384" actually ran
    lr=2e-4, min_lr=2e-5, warmup_steps=1000, schedule="cosine",
    weight_decay=0.1, betas=(0.9, 0.95), eps=1e-8, grad_clip=1.0,
    max_tokens="all",                 # "all" = one pass over train.bin (the full cleaned corpus); or an integer
    # plumbing
    eval_every_min=30, eval_windows=360, log_every=50, compile=True, fast_attention=True, seed=42,
    max_micro_batch=64,
)
FERTILITY_FILTER = False             # yours: keep ALL documents (multilingual + noisy OCR); True = Phase-1 rule (> 2.0 tok/word dropped)
MAX_DAYS = 7                          # safety stop only (the schedule is driven by max_tokens)
COST_PER_HOUR = None                  # optional: your instance $/h, for the projection in the probe

def _missing(d, prefix):
    return [prefix + k for k, v in d.items() if v is None and k not in ("attention_scale",)]
todo = _missing(MODEL, "MODEL.") + _missing(TRAIN, "TRAIN.")
assert not todo, "fill these first: " + ", ".join(todo)
assert TRAIN["batch_tokens"] % MODEL["max_seq_len"] == 0, "batch_tokens must be a multiple of max_seq_len"
assert os.path.exists(TOKENIZER), f"put legal_bpe.json at {TOKENIZER}"
sha = hashlib.sha256(open(TOKENIZER, "rb").read()).hexdigest()[:8]
print("tokenizer sha256", sha, "(Phase-1 legal BPE: 38f51f14)")
assert sha == "38f51f14", "this is not your Phase-1 tokenizer: the model would be unusable in the Phase-2+ pipeline"

def resolved_train():
    t = dict(TRAIN)
    if t["max_tokens"] == "all":
        man = os.path.join(DATA_DIR, "manifest.json")
        assert os.path.exists(man), "max_tokens='all' needs the data built first (section 4)"
        t["max_tokens"] = int(json.load(open(man))["train_tokens"])
    return t

def run_config(out_dir, probe_minutes=None):
    cfg = {"tag": TAG, "module": "arch3", "model": MODEL, "init_std": INIT_STD,
           "data_dir": DATA_DIR, "out_dir": out_dir, "tokenizer_path": TOKENIZER,
           "deadline_unix": time.time() + MAX_DAYS * 86400, "session_end_unix": time.time() + MAX_DAYS * 86400,
           "schedule_hours": None, "init_from": None,
           "resume_from": os.path.join(out_dir, "last.pt"),          # auto-resume if it exists
           "train": resolved_train(), "gen": {"max_new_tokens": 1, "decoding": {}}}
    if probe_minutes:
        cfg["probe"] = {"train_minutes": probe_minutes}
        cfg["resume_from"] = None
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, "config.json")
    json.dump(cfg, open(path, "w"), indent=2)
    return path
print("configuration OK")

## 2 — code files (plumbing; your arch3 unchanged)

In [ ]:
%%writefile a1_tokenizer.py
# Legal byte-level BPE (your A1 tokenizer, optimised)
# tokenizer.py
#
# Byte-level BPE with an end-of-word marker (A1 design), adapted for the CS781 legal corpus.
#
# Changes vs. the A1 version (original kept in Assignment3/a1_tokenizer.py):
#   * Tokens are integer ids backed by raw bytes, not strings. Previously the byte "<", "/",
#     "w", ">" could merge into a token identical to the "</w>" marker (or to "<pad>"), and
#     decode() would then insert spurious spaces. Ids make that collision impossible.
#   * Legal-text pre-tokenisation inside each whitespace word: letter runs, digit groups of
#     at most 3, and punctuation runs are merged separately ("s.302(1)," -> s . 302 ( 1 ) ,),
#     so "court", "court," and "court." share one vocabulary entry instead of three, and
#     years / section numbers / citations don't flood the vocabulary. Only the last piece of
#     each whitespace word carries </w>, so decode(encode(x)) == " ".join(x.split()).
#   * Incremental BPE training (pair counts + inverted index + lazy max-heap): each merge
#     only touches the words that contain the merged pair, instead of recounting every pair
#     in the corpus for every merge. 32k merges on a legal sample take minutes, not days.
#   * Encoding uses merge ranks (GPT-2 style, same result as applying merges in order) and a
#     bounded per-word cache (the old lru_cache on a method also cached `self` and never freed).
#   * Extra reserved special tokens for later phases (cls/sep/mask, span-corruption sentinels).
#   * encode_many() encodes many documents in parallel worker processes.
#
# Assignment 3 contract is unchanged: train_tokenizer(train_text, vocab_size) returns an object
# with encode(text) -> list[int], decode(ids) -> str and a vocab_size attribute.

import re
import os
import heapq
import string
import json
from collections import Counter, defaultdict


# ============================================================
# SPECIAL TOKENS
# ============================================================

N_SENTINELS = 25

SPECIALS = (
    ["<pad>", "<unk>", "<s>", "</s>", "<cls>", "<sep>", "<mask>"]
    + [f"<extra_id_{i}>" for i in range(N_SENTINELS)]
)   # 32 specials: ids 0..31

MIN_COUNT = 2

# Pieces inside one whitespace word: letter runs | digit groups (<= 3) | punctuation runs | underscores.
# The alternatives cover every non-space character, so "".join(pieces) == word always holds.
PRETOKENIZE = re.compile(r"[^\W\d_]+|\d{1,3}|[^\w\s]+|_+")

MAX_CACHE_WORDS = 2_000_000


# ============================================================
# TEXT CLEANING (A1 helpers, unchanged; not used by the BPE)
# ============================================================

def check_valid_en(word):
    is_en = False
    is_hi = False
    has_digit = False

    for ch in word:
        code = ord(ch)

        # English alphabet
        if 0x0041 <= code <= 0x005A or 0x0061 <= code <= 0x007A:
            is_en = True

        # Arabic numerals
        elif 0x0030 <= code <= 0x0039:
            has_digit = True

        # Devanagari numerals
        elif 0x0966 <= code <= 0x096F:
            has_digit = True

        # Devanagari
        elif 0x0900 <= code <= 0x097F:
            is_hi = True

    if has_digit:
        return False

    return is_en and not is_hi


def clean_text_and_tokenize_en(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    tokens = text.split()
    return [word for word in tokens if check_valid_en(word)]


# ============================================================
# BYTE-LEVEL BPE
# ============================================================

def split_word(word):
    """Whitespace word -> list of (piece, is_last_piece)."""
    pieces = PRETOKENIZE.findall(word) or [word]
    last = len(pieces) - 1
    return [(p, i == last) for i, p in enumerate(pieces)]


class ByteBPE:

    def __init__(self, vocab_size=16000):

        self.vocab_size = vocab_size

        # Id layout: specials | 256 bytes | </w> | one id per learned merge
        self.n_special = len(SPECIALS)
        self.byte_offset = self.n_special
        self.eow_id = self.byte_offset + 256

        self.merges = []            # list of (left_id, right_id); merge k creates id eow_id + 1 + k
        self._reset_tables()

    # --------------------------------------------------------
    # VOCAB TABLES
    # --------------------------------------------------------

    def _reset_tables(self):
        self.token_bytes = [b""] * self.n_special + [bytes([b]) for b in range(256)] + [b""]
        self.token_eow = [False] * (self.n_special + 256) + [True]
        for left, right in self.merges:
            self.token_bytes.append(self.token_bytes[left] + self.token_bytes[right])
            self.token_eow.append(self.token_eow[right])
        self.ranks = {pair: rank for rank, pair in enumerate(self.merges)}
        self.merge_to_id = {pair: self.eow_id + 1 + rank for rank, pair in enumerate(self.merges)}
        # decode table: bytes of each token, plus a space if it ends a word; specials decode to nothing
        self._decode_table = [
            b"" if i < self.n_special else tb + (b" " if eow else b"")
            for i, (tb, eow) in enumerate(zip(self.token_bytes, self.token_eow))
        ]
        self.itos = list(SPECIALS) + [self._display(i) for i in range(self.n_special, len(self.token_bytes))]
        self.stoi = {s: i for i, s in enumerate(self.itos)}
        self.vocab = self.itos
        self._cache = {}

    def _display(self, idx):
        """Human-readable token string (for inspection only; never used for encoding)."""
        text = self.token_bytes[idx].decode("utf-8", errors="backslashreplace")
        return text + ("</w>" if self.token_eow[idx] else "")

    @property
    def special_ids(self):
        return {tok: i for i, tok in enumerate(SPECIALS)}

    @property
    def eot_id(self):
        """Document separator used in the pretraining stream."""
        return SPECIALS.index("</s>")

    def get_vocab_size(self):
        return len(self.token_bytes)

    # --------------------------------------------------------
    # PIECE -> INITIAL SYMBOLS
    # --------------------------------------------------------

    def _piece_symbols(self, piece, is_last):
        symbols = [self.byte_offset + b for b in piece.encode("utf-8")]
        if is_last:
            symbols.append(self.eow_id)
        return symbols

    # --------------------------------------------------------
    # TRAIN BPE
    # --------------------------------------------------------

    @staticmethod
    def count_words(texts):
        """Whitespace-word frequencies of an iterable of texts."""
        counts = Counter()
        for text in texts:
            counts.update(str(text).split())
        return counts

    def fit(self, tokenized_corpus=None, target_vocab_size=None, min_count=MIN_COUNT,
            word_counts=None, min_word_freq=1, verbose=True):
        """Learn merges.

        tokenized_corpus: iterable of lists of whitespace words (A1/A3 interface), or
        word_counts:      a precomputed {whitespace word: frequency} mapping (faster for big corpora).
        min_word_freq:    ignore whitespace words rarer than this while learning merges (speed knob).
        """
        target = self.vocab_size if target_vocab_size is None else target_vocab_size
        self.vocab_size = target

        if word_counts is None:
            word_counts = Counter()
            for sentence_tokens in tokenized_corpus:
                word_counts.update(sentence_tokens)

        # whitespace words -> (piece, is_last) frequencies
        piece_freq = Counter()
        for word, count in word_counts.items():
            if count < min_word_freq:
                continue
            for piece in split_word(word):
                piece_freq[piece] += count

        words = [self._piece_symbols(p, last) for p, last in piece_freq]
        freqs = list(piece_freq.values())
        del piece_freq

        if verbose:
            print("=" * 60)
            print("BUILDING BYTE-LEVEL BPE")
            print("=" * 60)
            print(f"Unique pre-token pieces: {len(words):,}")
            print(f"Initial vocabulary size: {self.eow_id + 1:,}")
            print(f"Target vocabulary size: {target:,}")

        # pair -> total count, pair -> set of word indices containing it
        pair_counts = defaultdict(int)
        where = defaultdict(set)
        for wi, (symbols, f) in enumerate(zip(words, freqs)):
            for pair in zip(symbols, symbols[1:]):
                pair_counts[pair] += f
                where[pair].add(wi)

        # max-heap on count with deterministic tie-break on ids; stale entries are skipped lazily
        heap = [(-c, pair) for pair, c in pair_counts.items()]
        heapq.heapify(heap)

        self.merges = []
        next_id = self.eow_id + 1

        while next_id < target and heap:
            neg, pair = heapq.heappop(heap)
            current = pair_counts.get(pair, 0)
            if current != -neg:
                if current > 0:
                    heapq.heappush(heap, (-current, pair))
                continue
            if current < min_count:
                if verbose:
                    print(f"\nStopping: best pair frequency = {current}")
                break

            left, right = pair
            new_id = next_id
            next_id += 1
            self.merges.append(pair)
            touched = Counter()

            for wi in list(where.pop(pair, ())):
                symbols, f = words[wi], freqs[wi]
                # remove this word's old pairs
                for p in zip(symbols, symbols[1:]):
                    pair_counts[p] -= f
                    touched[p] += 0
                    s = where.get(p)
                    if s is not None:
                        s.discard(wi)
                # merge left-to-right
                merged, i, n = [], 0, len(symbols)
                while i < n:
                    if i < n - 1 and symbols[i] == left and symbols[i + 1] == right:
                        merged.append(new_id)
                        i += 2
                    else:
                        merged.append(symbols[i])
                        i += 1
                words[wi] = merged
                # add the new pairs
                for p in zip(merged, merged[1:]):
                    pair_counts[p] += f
                    touched[p] += 0
                    where[p].add(wi)

            pair_counts.pop(pair, None)
            for p in touched:
                c = pair_counts.get(p, 0)
                if c <= 0:
                    pair_counts.pop(p, None)
                    where.pop(p, None)
                elif new_id in p:
                    heapq.heappush(heap, (-c, p))   # only pairs with the new symbol can grow

            if verbose and len(self.merges) % 1000 == 0:
                print(f"Merges: {len(self.merges):,} | Vocab: {next_id:,} | Latest pair count: {current:,}")

        self._reset_tables()

        if verbose:
            print("\n" + "=" * 60)
            print("BPE TRAINING COMPLETE")
            print("=" * 60)
            print(f"Final vocabulary size: {self.get_vocab_size():,}")
            print(f"Total merges learned: {len(self.merges):,}")

        return self

    # --------------------------------------------------------
    # ENCODE
    # --------------------------------------------------------

    def _bpe(self, symbols):
        """Repeatedly merge the lowest-rank adjacent pair (== applying merges in learned order)."""
        ranks = self.ranks
        while len(symbols) > 1:
            best, best_rank = None, None
            for pair in zip(symbols, symbols[1:]):
                r = ranks.get(pair)
                if r is not None and (best_rank is None or r < best_rank):
                    best, best_rank = pair, r
            if best is None:
                break
            new_id = self.merge_to_id[best]
            left, right = best
            merged, i, n = [], 0, len(symbols)
            while i < n:
                if i < n - 1 and symbols[i] == left and symbols[i + 1] == right:
                    merged.append(new_id)
                    i += 2
                else:
                    merged.append(symbols[i])
                    i += 1
            symbols = merged
        return symbols

    def encode_word(self, word):
        ids = self._cache.get(word)
        if ids is None:
            ids = []
            for piece, is_last in split_word(word):
                ids.extend(self._bpe(self._piece_symbols(piece, is_last)))
            ids = tuple(ids)
            if len(self._cache) >= MAX_CACHE_WORDS:
                self._cache.clear()
            self._cache[word] = ids
        return ids

    def encode_tokens(self, sentence_tokens):
        ids = []
        for word in sentence_tokens:
            ids.extend(self.encode_word(word))
        return ids

    def encode(self, text):
        """text -> list of integer ids (whitespace is normalised to single spaces)."""
        cache = self._cache
        ids = []
        for word in str(text).split():
            w = cache.get(word)
            if w is None:
                w = self.encode_word(word)
            ids.extend(w)
        return ids

    def pool(self, workers=None):
        """Process pool whose workers each hold a copy of this tokenizer (and keep their word cache).
        Use with the module-level worker functions, e.g. pool.imap(encode_array_worker, texts)."""
        import multiprocessing as mp
        workers = workers or os.cpu_count() or 1
        return mp.get_context("fork").Pool(workers, initializer=_worker_init, initargs=(self,))

    def encode_many(self, texts, workers=None, chunksize=64):
        """Encode many documents in parallel processes; yields lists of ids in input order."""
        with self.pool(workers) as pool:
            yield from pool.imap(_worker_encode, texts, chunksize=chunksize)

    # --------------------------------------------------------
    # DECODE
    # --------------------------------------------------------

    def decode(self, ids):
        table = self._decode_table
        n = len(table)
        out = b"".join(table[int(i)] if 0 <= int(i) < n else b"" for i in ids)
        return out.decode("utf-8", errors="replace").strip()

    def ends_word(self, idx):
        """True if token idx closes a whitespace word (used to count generated words)."""
        return self.token_eow[idx]

    # --------------------------------------------------------
    # SAVE / LOAD
    # --------------------------------------------------------

    def to_json(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "format": "legal-bytebpe-v2",
                    "specials": SPECIALS,
                    "merges": self.merges,
                    "vocab_size": self.get_vocab_size(),
                    "itos": self.itos,          # readable, informational only
                },
                f,
                ensure_ascii=False,
            )
        print(f"Saved tokenizer to: {path}")

    @classmethod
    def from_json(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if data.get("format") != "legal-bytebpe-v2":
            raise ValueError(f"{path} was not written by this tokenizer version")
        if data["specials"] != SPECIALS:
            raise ValueError("special tokens differ from this tokenizer version")
        tokenizer = cls(vocab_size=data["vocab_size"])
        tokenizer.merges = [tuple(pair) for pair in data["merges"]]
        tokenizer._reset_tables()
        return tokenizer


# worker-process helpers for encode_many (module level so they pickle)
_WORKER_TOKENIZER = None


def _worker_init(tokenizer):
    global _WORKER_TOKENIZER
    _WORKER_TOKENIZER = tokenizer


def _worker_encode(text):
    return _WORKER_TOKENIZER.encode(text)


def encode_array_worker(text):
    """(uint16 id array, whitespace word count) for one document; for tokenizer.pool().imap."""
    import numpy as np
    return np.asarray(_WORKER_TOKENIZER.encode(text), dtype=np.uint16), text.count(" ") + 1


# ============================================================
# ASSIGNMENT 3 TRAINING FUNCTION
# ============================================================

def train_tokenizer(train_text: str, vocab_size: int):
    text = str(train_text)
    print(f"Training tokenizer on {len(text):,} characters")
    tokenizer = ByteBPE(vocab_size=vocab_size)
    tokenizer.fit(
        word_counts=ByteBPE.count_words(text.splitlines()),
        target_vocab_size=vocab_size,
        min_count=MIN_COUNT,
    )
    return tokenizer


# ============================================================
# SIMPLE SMOKE TEST
# ============================================================

if __name__ == "__main__":

    print("=" * 60)
    print("TOKENIZER SMOKE TEST")
    print("=" * 60)

    train_text = """
    the court has jurisdiction over this matter under Section 302(1) of the Code.
    the court shall consider the evidence, vide AIR 1966 SC 1882, dated 18.12.1961.
    the evidence presented before the court </w> <pad> was 12,345 pages.
    """

    tokenizer = train_tokenizer(train_text, vocab_size=400)

    for test_text in [
        "the court has jurisdiction",
        "Section 302(1), AIR 1966 SC 1882 </w><pad> naïve — ₹5,00,000/- (u/s 138)",
    ]:
        ids = tokenizer.encode(test_text)
        decoded = tokenizer.decode(ids)
        print("\nInput:  ", test_text)
        print("Ids:    ", ids)
        print("Pieces: ", [tokenizer.itos[i] for i in ids])
        print("Decoded:", decoded)
        assert decoded == " ".join(test_text.split()), "round trip failed"

    print("\nVocabulary size:", tokenizer.get_vocab_size())

    tokenizer.to_json("bpe_smoke_test.json")
    loaded = ByteBPE.from_json("bpe_smoke_test.json")
    assert loaded.encode(test_text) == tokenizer.encode(test_text)
    os.remove("bpe_smoke_test.json")

    print("\n" + "=" * 60)
    print("SMOKE TEST PASSED")
    print("=" * 60)

In [ ]:
%%writefile arch3.py
# Architecture 2 option — rope.py + attention3.py + decoder3.py (your code: GQA + RoPE)
# ======== rope.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Rotary Positional Embeddings, implemented from scratch."""
import torch


def build_rope_cache(max_seq_len: int, d_head: int, base: float = 10000.0):
    """
    Returns (cos, sin) tensors of shape (max_seq_len, d_head) ready to be applied to
    query/key tensors of head dimension d_head. d_head must be even. This scaffold uses
    the split-half rotate_half convention: channel i is paired with channel i+d_head/2.
    """
    assert d_head % 2 == 0, "RoPE requires an even head dimension"
    # TODO: compute inverse frequencies: 1 / base^(2i/d_head) for i in [0, d_head/2)
    inverse = [1/(base**(2*i/d_head)) for i in range(0,d_head//2)]
    inverse = torch.tensor(inverse, dtype=torch.float32)  # FIX (Claude): torch.outer needs a tensor, not a list
    # TODO: outer product of positions [0..max_seq_len) with inverse frequencies -> (T, d_head/2)
    pos = torch.arange(max_seq_len, dtype=torch.float32)  # FIX (Claude): float, to match `inverse` in torch.outer
    outer = torch.outer(pos,inverse)
    # TODO: construct angles = torch.cat([freqs, freqs], dim=-1) so the shape
    #       becomes (T, d_head) and matches the split-half rotate_half convention.
    #       Do NOT use repeat_interleave here; it produces a different channel layout.
    #       Then return angles.cos(), angles.sin().
    angles = torch.cat([outer,outer],dim=-1)
    return angles.cos() , angles.sin()
    


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Split the last dim in half and return (-x2, x1) concatenated -- the 90-degree
    rotation step used by the "rotate_half" formulation of RoPE."""
    x1,x2 = torch.chunk(x,2,dim=-1)
    x2=-x2
    x = torch.cat([x2,x1],dim=-1)
    return x



def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    x: (B, n_heads, T, d_head) -- a query or key tensor.
    cos, sin: (T, d_head), sliced from build_rope_cache's output for the current T.
    Returns the rotated tensor, same shape as x.
    Formula: x_rotated = x * cos + rotate_half(x) * sin
    """
    # TODO: broadcast cos/sin to (1, 1, T, d_head) and apply the formula above
    cos = torch.unsqueeze(cos,dim=0)
    cos = torch.unsqueeze(cos,dim=0)
    sin = torch.unsqueeze(sin,dim=0)
    sin = torch.unsqueeze(sin,dim=0)
    x_rotated = x*cos+rotate_half(x)*sin
    
    return x_rotated


# ======== attention3.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            q = apply_rope(q,self.rope_cos[:T],self.rope_sin[:T])
            k = apply_rope(k,self.rope_cos[:T],self.rope_sin[:T])
        

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # TODO: same attention as before, but compute scores as
        #       (q @ k.transpose(-2, -1)) * self.attention_scale before masking/softmax
        attention_scores = (q@k.transpose(-2,-1))*self.attention_scale
       
        attention_scores = attention_scores.masked_fill(
            ~self.causal_mask[:T,:T],float("-inf")
        )
       
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
    
        weighted_sum = attention_weights@v
   
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output


# ======== decoder3.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Decoder-only Transformer LM -- Block now passes use_rope through to attention, and
DecoderLM only builds/adds the sinusoidal table when pos_emb_type == "sinusoidal"
(everything else unchanged from Question 2)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    PE = torch.zeros(max_len, d_model)

    for pos in range(max_len):
        PE[pos, 0::2] = torch.sin(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
        PE[pos, 1::2] = torch.cos(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
    return PE  # unchanged from Question 1


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg=cfg
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        if self.cfg.norm_first is True:
            y1 = self.ln1(x)
            y2 = self.attn(y1)+x
            y3 = self.ln2(y2)
            y4 = self.ffn(y3)+y2
            output=y4
        else:
            y1 = self.attn(x)
            y2 = self.ln1(y1+x)
            y3 = self.ffn(y2)
            y4 = self.ln2(y3+y2)
            output=y4
            
        
        # and post-norm when norm_first=False.
        return output


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        x = self.tok_emb(idx)
        if(self.cfg.pos_emb_type == "sinusoidal"):
            x=x+self.pos_emb[:T]        # TODO: run x through every block, then ln_f, then self.head
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        x = self.head(x)
        
        return x  # unchanged from Question 1 (just remember: only add
        # self.pos_emb when self.cfg.pos_emb_type == "sinusoidal")

In [ ]:
%%writefile fastattn.py
# Plumbing: fused SDPA attention + KV cache + generation around your modules
"""Efficiency plumbing around the hand-written A3 models (written with Claude; declare it).

Nothing here changes an architecture: parameters, shapes and the maths of every model stay
exactly as defined in arch1/arch2/arch3. `patch_model` swaps each attention module's forward
for one that computes the *same* function with fused kernels, and adds an optional KV cache
for generation. `check_equivalence` in the notebook verifies outputs match the original code.

* Training: F.scaled_dot_product_attention(is_causal=True). On T4 (sm_75) PyTorch cannot use
  the FlashAttention-2 kernel (needs sm_80+); it dispatches to the memory-efficient fused
  kernel, which gives the same benefit: no (T x T) score matrix materialised in HBM.
* Generation: per-layer KV cache (keys/values for MHA/GQA, the compressed latent for MLA),
  prefill once, then one token per step.
"""
import sys
import types
from contextlib import contextmanager

import torch
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# attention forward (drop-in for all three hand-written attention classes)
# ---------------------------------------------------------------------------

def _sdpa_supports_gqa():
    """PyTorch >= 2.5: scaled_dot_product_attention(..., enable_gqa=True) attends with fewer K/V heads directly."""
    try:
        q = torch.zeros(1, 4, 2, 8)
        kv = torch.zeros(1, 2, 2, 8)
        F.scaled_dot_product_attention(q, kv, kv, enable_gqa=True)
        return True
    except (TypeError, RuntimeError):
        return False


SDPA_GQA = _sdpa_supports_gqa()


def _sdpa(q, k, v, causal, dropout_p, scale, gqa=False):
    if gqa:
        return F.scaled_dot_product_attention(q, k, v, is_causal=causal, dropout_p=dropout_p, scale=scale,
                                              enable_gqa=True)
    return F.scaled_dot_product_attention(q, k, v, is_causal=causal, dropout_p=dropout_p, scale=scale)


def _fast_attention_forward(self, x):
    B, T, C = x.shape
    cache = getattr(self, "_kv_cache", None)
    start = cache["pos"] if cache is not None else 0
    n_heads, d_head = self.n_heads, self.d_head
    dropout_p = self.attn_dropout.p if self.training else 0.0

    q = self.q_proj(x).view(B, T, n_heads, d_head).transpose(1, 2)

    if hasattr(self, "kv_down_proj"):                       # MLA: cache the shared latent
        latent = self.kv_down_proj(x)
        if cache is not None:
            cache["latent"][:, start:start + T] = latent
            latent = cache["latent"][:, :start + T]
        S = latent.shape[1]
        k = self.k_up_proj(latent).view(B, S, n_heads, d_head).transpose(1, 2)
        v = self.v_up_proj(latent).view(B, S, n_heads, d_head).transpose(1, 2)
    else:                                                   # MHA / GQA (+ optional RoPE)
        n_kv = getattr(self, "n_kv_heads", n_heads)
        k = self.k_proj(x).view(B, T, n_kv, d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, n_kv, d_head).transpose(1, 2)
        if getattr(self, "use_rope", False):
            cos = self.rope_cos[start:start + T]
            sin = self.rope_sin[start:start + T]
            q = self._apply_rope(q.float(), cos, sin).to(v.dtype)
            k = self._apply_rope(k.float(), cos, sin).to(v.dtype)
        if cache is not None:
            cache["k"][:, :, start:start + T] = k
            cache["v"][:, :, start:start + T] = v
            k = cache["k"][:, :, :start + T]
            v = cache["v"][:, :, :start + T]
        n_rep = n_heads // n_kv
        use_gqa = n_rep > 1 and SDPA_GQA            # native GQA: no repeated copies of K/V in memory
        if n_rep > 1 and not use_gqa:
            k = k.repeat_interleave(n_rep, dim=1)
            v = v.repeat_interleave(n_rep, dim=1)

    # prefill (start == 0) is causal; a single decode step attends to the whole cache
    causal = T > 1
    if causal and start > 0:
        raise ValueError("multi-token steps are only supported from an empty cache")
    y = _sdpa(q, k, v, causal, dropout_p, self.attention_scale, gqa=locals().get("use_gqa", False))
    y = y.transpose(1, 2).reshape(B, T, n_heads * d_head)
    return self.resid_dropout(self.out_proj(y))


def attention_modules(model):
    return [block.attn for block in model.blocks]


def patch_model(model):
    """Swap every attention forward for the fused/cached version (same parameters)."""
    for attn in attention_modules(model):
        attn._kv_cache = None
        if getattr(attn, "use_rope", False):
            # the user's own apply_rope, from the module that defines this attention class
            attn._apply_rope = sys.modules[type(attn).__module__].apply_rope
        attn.forward = types.MethodType(_fast_attention_forward, attn)
    return model


# ---------------------------------------------------------------------------
# KV cache
# ---------------------------------------------------------------------------

@contextmanager
def kv_cache(model, batch_size, max_len, dtype):
    """Allocate a per-layer cache. Inside the context call `step(model, ids)` repeatedly."""
    dev = next(model.parameters()).device
    caches = []
    for attn in attention_modules(model):
        c = {"pos": 0}
        if hasattr(attn, "kv_down_proj"):
            c["latent"] = torch.empty(batch_size, max_len, attn.kv_latent_dim, device=dev, dtype=dtype)
        else:
            n_kv = getattr(attn, "n_kv_heads", attn.n_heads)
            shape = (batch_size, n_kv, max_len, attn.d_head)
            c["k"] = torch.empty(shape, device=dev, dtype=dtype)
            c["v"] = torch.empty(shape, device=dev, dtype=dtype)
        attn._kv_cache = c
        caches.append(c)
    full_pe = getattr(model, "pos_emb", None)
    model._fast_state = {"caches": caches, "full_pe": full_pe}
    try:
        yield
    finally:
        for attn in attention_modules(model):
            attn._kv_cache = None
        if full_pe is not None:
            model.pos_emb = full_pe
        del model._fast_state


def step(model, ids):
    """Run `ids` (B, T) through the model at the current cache position; returns logits (B, T, V)."""
    state = model._fast_state
    pos = state["caches"][0]["pos"]
    if state["full_pe"] is not None:
        # DecoderLM.forward adds self.pos_emb[:T]; shift the table so rows pos..pos+T are used
        model.pos_emb = state["full_pe"][pos:]
    logits = model(ids)
    for c in state["caches"]:
        c["pos"] += ids.shape[1]
    return logits


# ---------------------------------------------------------------------------
# sampling + word-budgeted generation
# ---------------------------------------------------------------------------

def sample_next(logits, temperature, top_k, top_p, banned_ids, prev_ids=None, repetition_penalty=1.0):
    logits = torch.nan_to_num(logits.float(), nan=-1e4)   # a diverged checkpoint must not crash sampling
    if banned_ids is not None:
        logits[:, banned_ids] = -float("inf")
    if repetition_penalty != 1.0 and prev_ids is not None:
        seen = torch.gather(logits, 1, prev_ids)
        seen = torch.where(seen > 0, seen / repetition_penalty, seen * repetition_penalty)
        logits = logits.scatter(1, prev_ids, seen)
    if not temperature:
        return logits.argmax(-1)
    logits = logits / temperature
    if top_k:
        kth = torch.topk(logits, top_k, dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    if top_p and top_p < 1.0:
        sorted_logits, idx = torch.sort(logits, descending=True)
        probs = sorted_logits.softmax(-1)
        drop = probs.cumsum(-1) - probs > top_p          # keep the smallest set with mass >= top_p
        sorted_logits = sorted_logits.masked_fill(drop, -float("inf"))
        logits = torch.full_like(logits, -float("inf")).scatter(1, idx, sorted_logits)
    return torch.multinomial(logits.softmax(-1), 1).squeeze(1)


def ngram_ban(seq, n, vocab):
    """(B, V) bool: tokens that would complete an n-gram (in token ids) already present in `seq` (B, S).
    For each row, every earlier occurrence of the last n-1 tokens bans the token that followed it."""
    B, S = seq.shape
    if n < 2 or S < n:
        return torch.zeros(B, vocab, dtype=torch.bool, device=seq.device)
    grams = seq.unfold(1, n, 1)                                   # (B, S-n+1, n)
    hit = (grams[:, :, :-1] == seq[:, None, S - n + 1:]).all(-1)  # earlier (n-1)-gram == current suffix
    cnt = torch.zeros(B, vocab, device=seq.device).scatter_add_(1, grams[:, :, -1], hit.float())
    return cnt > 0


@torch.inference_mode()
def generate_words(model, tokenizer, prompts, *, max_seq_len, target_words, max_new_tokens,
                   batch_size, temperature, top_k, top_p, repetition_penalty, device, amp_dtype,
                   log_every=10, no_repeat_ngram=0, ngram_scope="output"):
    """no_repeat_ngram = n > 1 forbids repeating any n-gram of TOKEN ids (HF no_repeat_ngram_size), counted over
    the generated text only (ngram_scope="output") or over prompt + generated text ("all")."""
    assert ngram_scope in ("output", "all"), ngram_scope
    """Continue each prompt by `target_words` whitespace words. Returns list[str] in input order.

    Prompts are bucketed by token length so a batch needs no padding: every prompt in a batch
    keeps its last L tokens, L = shortest prompt in the batch (capped so prompt + new tokens fit
    max_seq_len). Words are counted from the tokenizer's end-of-word flag, so we stop as soon as
    every row has produced target_words complete words.
    """
    model.eval()
    eow = torch.tensor(tokenizer.token_eow, device=device, dtype=torch.bool)
    has_bytes = torch.tensor([len(b) > 0 for b in tokenizer.token_bytes], device=device, dtype=torch.bool)
    banned = torch.arange(tokenizer.n_special, device=device)     # never emit pad/unk/</s>/...
    encoded = [tokenizer.encode(p) for p in prompts]
    order = sorted(range(len(prompts)), key=lambda i: len(encoded[i]))
    max_prompt = max_seq_len - max_new_tokens
    assert max_prompt > 0, "max_new_tokens must be smaller than max_seq_len"
    results = [None] * len(prompts)

    for b, start in enumerate(range(0, len(order), batch_size)):
        idxs = order[start:start + batch_size]
        L = min(min(len(encoded[i]) for i in idxs), max_prompt)
        ids = torch.tensor([encoded[i][-L:] for i in idxs], device=device, dtype=torch.long)
        B = len(idxs)
        words = torch.zeros(B, dtype=torch.int32, device=device)
        in_word = torch.zeros(B, dtype=torch.bool, device=device)     # prompts end on a full word
        done = torch.zeros(B, dtype=torch.bool, device=device)
        out = []
        with torch.autocast("cuda", dtype=amp_dtype, enabled=device.type == "cuda"), \
                kv_cache(model, B, L + max_new_tokens, amp_dtype if device.type == "cuda" else torch.float32):
            logits = step(model, ids)[:, -1]
            history = ids
            for t in range(max_new_tokens):
                # a bare </w> right after a finished word never occurs in encoded text: forbid it
                logits = logits.float().clone()
                logits[~in_word, tokenizer.eow_id] = -float("inf")
                logits[:, len(tokenizer.token_bytes):] = -float("inf")   # ids the tokenizer can't decode
                if no_repeat_ngram and no_repeat_ngram > 1:
                    seq = history if ngram_scope == "all" else (torch.stack(out, 1) if out else ids[:, :0])
                    ban = ngram_ban(seq, no_repeat_ngram, logits.size(1))
                    ban &= ~(ban | torch.isinf(logits)).all(-1, keepdim=True)  # never leave a row with no option
                    logits = logits.masked_fill(ban, -float("inf"))
                nxt = sample_next(logits, temperature, top_k, top_p, banned,
                                  history if repetition_penalty != 1.0 else None, repetition_penalty)
                out.append(nxt)
                # a word ends on an end-of-word token that closes something non-empty
                open_ = in_word | has_bytes[nxt]
                words += (eow[nxt] & open_ & ~done).int()
                in_word = open_ & ~eow[nxt]
                done |= words >= target_words
                if bool(done.all()):
                    break
                if repetition_penalty != 1.0 or (no_repeat_ngram and ngram_scope == "all"):
                    history = torch.cat([history, nxt[:, None]], 1)
                logits = step(model, nxt[:, None])[:, -1]
        gen = torch.stack(out, 1).tolist()
        for row, i in enumerate(idxs):
            text = tokenizer.decode(gen[row])
            results[i] = " ".join(text.split()[:target_words])
        if log_every and (b + 1) % log_every == 0:
            print(f"  generated {min(start + batch_size, len(order))}/{len(order)}", flush=True)
    return results


@torch.inference_mode()
def generate_beam(model, tokenizer, prompts, *, max_seq_len, target_words, max_new_tokens, batch_size,
                  beam_size, device, amp_dtype, no_repeat_ngram=0, ngram_scope="output", length_penalty=1.0,
                  log_every=10):
    """Beam search with the same word budget / token constraints as generate_words.
    Beams are ranked by sum(log p) / length**length_penalty (1.0 = mean log-prob per token). A beam that has
    produced `target_words` words is frozen (score and length fixed) and keeps competing as a finished
    hypothesis. Optional n-gram blocking per beam (same rule as generate_words). Returns list[str] in input order."""
    assert ngram_scope in ("output", "all"), ngram_scope
    model.eval()
    W = int(beam_size)
    eow = torch.tensor(tokenizer.token_eow, device=device, dtype=torch.bool)
    has_bytes = torch.tensor([len(b) > 0 for b in tokenizer.token_bytes], device=device, dtype=torch.bool)
    n_decodable = len(tokenizer.token_bytes)
    encoded = [tokenizer.encode(p) for p in prompts]
    order = sorted(range(len(prompts)), key=lambda i: len(encoded[i]))
    max_prompt = max_seq_len - max_new_tokens
    per_batch = max(1, batch_size // W)
    results = [None] * len(prompts)
    neg_inf = -float("inf")

    for b, start in enumerate(range(0, len(order), per_batch)):
        idxs = order[start:start + per_batch]
        P = len(idxs)
        R = P * W
        L = min(min(len(encoded[i]) for i in idxs), max_prompt)
        prompt_ids = torch.tensor([encoded[i][-L:] for i in idxs], device=device, dtype=torch.long)
        ids = prompt_ids.repeat_interleave(W, 0)                                   # (R, L): row = p * W + beam
        words = torch.zeros(R, dtype=torch.int32, device=device)
        in_word = torch.zeros(R, dtype=torch.bool, device=device)
        done = torch.zeros(R, dtype=torch.bool, device=device)
        cum = torch.zeros(R, device=device)
        length = torch.zeros(R, device=device)
        gen = torch.zeros(R, 0, dtype=torch.long, device=device)
        with torch.autocast("cuda", dtype=amp_dtype, enabled=device.type == "cuda"), \
                kv_cache(model, R, L + max_new_tokens, amp_dtype if device.type == "cuda" else torch.float32):
            logits = step(model, ids)[:, -1]
            for t in range(max_new_tokens):
                lp = torch.nan_to_num(logits.float(), nan=-1e4)
                lp[:, :tokenizer.n_special] = neg_inf
                lp[~in_word, tokenizer.eow_id] = neg_inf
                lp[:, n_decodable:] = neg_inf
                if no_repeat_ngram and no_repeat_ngram > 1:
                    seq = gen if ngram_scope == "output" else torch.cat([ids, gen], 1)
                    ban = ngram_ban(seq, no_repeat_ngram, lp.size(1))
                    ban &= ~(ban | torch.isinf(lp)).all(-1, keepdim=True)
                    lp = lp.masked_fill(ban, neg_inf)
                lp = lp.log_softmax(-1)
                V = lp.size(1)
                # candidate scores (length-normalised); finished beams keep their score via one frozen continuation
                cand = (cum[:, None] + lp) / (length[:, None] + 1).pow(length_penalty)
                frozen_tok = lp.argmax(-1)
                frozen = torch.full_like(cand, neg_inf)
                frozen.scatter_(1, frozen_tok[:, None], (cum / length.clamp(min=1).pow(length_penalty))[:, None])
                cand = torch.where(done[:, None], frozen, cand)
                if t == 0:                                                         # identical beams: expand beam 0 only
                    cand.view(P, W, V)[:, 1:] = neg_inf
                top_s, top_i = cand.view(P, W * V).topk(W, dim=-1)                 # (P, W)
                parent = (top_i // V) + torch.arange(P, device=device)[:, None] * W
                parent, tok = parent.reshape(-1), (top_i % V).reshape(-1)
                was_done = done[parent]
                # reorder beam state + KV cache to the chosen parents
                for c in model._fast_state["caches"]:
                    pos = c["pos"]
                    c["k"][:, :, :pos] = c["k"][parent, :, :pos]
                    c["v"][:, :, :pos] = c["v"][parent, :, :pos]
                words, in_word, done = words[parent], in_word[parent], done[parent].clone()
                cum = torch.where(was_done, cum[parent], cum[parent] + lp[parent, tok])
                length = torch.where(was_done, length[parent], length[parent] + 1)
                gen = torch.cat([gen[parent], tok[:, None]], 1)
                open_ = in_word | has_bytes[tok]
                words = words + (eow[tok] & open_ & ~was_done).int()
                in_word = open_ & ~eow[tok]
                done = was_done | (words >= target_words)
                if bool(done.all()):
                    break
                logits = step(model, tok[:, None])[:, -1]
        score = (cum / length.clamp(min=1).pow(length_penalty)).view(P, W)
        score = torch.where(done.view(P, W), score, score - 1e6)                     # prefer finished beams
        best = score.argmax(-1) + torch.arange(P, device=device) * W
        rows = gen[best].tolist()
        for row, i in enumerate(idxs):
            text = tokenizer.decode(rows[row])
            results[i] = " ".join(text.split()[:target_words])
        if log_every and (b + 1) % log_every == 0:
            print(f"  beam generated {min(start + per_batch, len(order))}/{len(order)}", flush=True)
    return results


# ---------------------------------------------------------------------------
# verification helpers (run in the notebook before training)
# ---------------------------------------------------------------------------

@torch.no_grad()
def check_equivalence(make_model, vocab_size, seq_len=48, rtol=1e-4):
    """Original forward vs patched forward vs KV-cached incremental decoding, fp32 on CPU.
    Returns max |difference| relative to the largest |logit| of the original model."""
    torch.manual_seed(0)
    ref = make_model().eval()
    fast = make_model().eval()
    fast.load_state_dict(ref.state_dict())
    patch_model(fast)
    idx = torch.randint(0, vocab_size, (2, seq_len))
    a = ref(idx)
    scale = a.abs().max().item()
    err_full = (a - fast(idx)).abs().max().item() / scale
    with kv_cache(fast, 2, seq_len, torch.float32):
        pre = step(fast, idx[:, :seq_len // 2])
        steps = [step(fast, idx[:, t:t + 1]) for t in range(seq_len // 2, seq_len)]
    err_cache = (a - torch.cat([pre] + steps, 1)).abs().max().item() / scale
    assert err_full < rtol and err_cache < rtol, (err_full, err_cache)
    return err_full, err_cache

In [ ]:
%%writefile build_data.py
#!/usr/bin/env python3
"""Tokenize the WHOLE Phase-1 legal corpus with your legal_bpe.json into train.bin / val.bin (plumbing; declare it).

  HF_TOKEN=... python build_data.py --tokenizer legal_bpe.json --out data
  python build_data.py --tokenizer legal_bpe.json --out data --local-dir corpus/   # files already downloaded

Cleaning: whitespace normalised, documents shorter than --min-words dropped, exact duplicates (hash of the first 2000
chars) dropped, and (unless --no-fertility-filter) non-Supreme-Court documents with fertility > --max-fertility
tokens/word dropped (garbled OCR, and in practice most non-English text with this tokenizer). Unlike Phase 1 there is no mixture / repetition: every remaining document
of every source is used once. A hash-chosen --val-percent of Supreme Court documents is held out as val.bin
(Supreme Court because the test prompts are ~93% SC).

Output (uint16 token ids, each document followed by </s>):
  data/train.bin   all sources concatenated (training samples random windows, so order does not matter)
  data/val.bin     held-out SC documents
  data/manifest.json   per-source docs / tokens / drops, tokenizer sha256, build time
Resumable: each source is written to data/parts/<source>.bin with a .done marker; finished sources are skipped
(cross-source duplicate removal then only covers the sources built in the current invocation).
"""
import argparse
import hashlib
import json
import os
import shutil
import sys
import time
from collections import Counter

import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import a1_tokenizer                      # noqa: E402
from a1_tokenizer import ByteBPE         # noqa: E402

REPO = "Exploration-Lab/CS-781-Capstone"
FILES = {
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
}


def parse_line(line):
    line = line.strip().rstrip(",")
    if not line.startswith("{"):
        return None
    try:
        return json.loads(line)
    except json.JSONDecodeError:
        return None


def iter_texts(path):
    """Stream the "text" field of a one-record-per-line JSON array file (no full load into memory)."""
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rec = parse_line(line)
            if rec and rec.get("text"):
                yield rec["text"]


SCRIPTS = [("devanagari", 0x0900, 0x097F), ("bengali", 0x0980, 0x09FF), ("gurmukhi", 0x0A00, 0x0A7F),
           ("gujarati", 0x0A80, 0x0AFF), ("odia", 0x0B00, 0x0B7F), ("tamil", 0x0B80, 0x0BFF), ("telugu", 0x0C00, 0x0C7F),
           ("kannada", 0x0C80, 0x0CFF), ("malayalam", 0x0D00, 0x0D7F), ("arabic_urdu", 0x0600, 0x06FF)]


def dominant_script(text, n=2000):
    """Most frequent script among the letters of the first n characters ("latin", one of SCRIPTS, or "other")."""
    counts = {}
    for ch in text[:n]:
        if not ch.isalpha():
            continue
        o = ord(ch)
        if o < 0x250:
            k = "latin"
        else:
            k = next((name for name, lo, hi in SCRIPTS if lo <= o <= hi), "other")
        counts[k] = counts.get(k, 0) + 1
    return max(counts, key=counts.get) if counts else "none"


def doc_hash(text):
    return int(hashlib.md5(text[:2000].encode("utf-8", errors="ignore")).hexdigest(), 16)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--tokenizer", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--local-dir", default=None, help="folder with the four train_*.json files (skip download)")
    ap.add_argument("--sources", nargs="+", default=list(FILES))
    ap.add_argument("--min-words", type=int, default=100)
    ap.add_argument("--max-fertility", type=float, default=2.0)
    ap.add_argument("--no-fertility-filter", action="store_true",
                    help="keep every document regardless of tokens/word (non-English scripts and noisy OCR included)")
    ap.add_argument("--val-percent", type=int, default=2)
    ap.add_argument("--workers", type=int, default=os.cpu_count() or 4)
    ap.add_argument("--max-docs", type=int, default=None, help="smoke tests only: docs per source")
    ap.add_argument("--dry-stats", action="store_true",
                    help="only count documents/tokens per script (kept vs dropped); writes no token files")
    args = ap.parse_args()

    t_all = time.time()
    os.makedirs(os.path.join(args.out, "parts"), exist_ok=True)
    tok = ByteBPE.from_json(args.tokenizer)
    tok_sha = hashlib.sha256(open(args.tokenizer, "rb").read()).hexdigest()
    V, EOT = tok.get_vocab_size(), tok.eot_id
    assert V <= 65535, "uint16 storage needs vocab_size <= 65535"
    print(f"tokenizer {args.tokenizer}: vocab {V}, </s> id {EOT}, sha256 {tok_sha[:8]} (Phase-1 legal BPE: 38f51f14)")

    paths = {}
    for name in args.sources:
        if args.local_dir:
            paths[name] = os.path.join(args.local_dir, os.path.basename(FILES[name]))
        else:
            from huggingface_hub import hf_hub_download
            paths[name] = hf_hub_download(repo_id=REPO, filename=FILES[name], repo_type="dataset",
                                          token=os.environ.get("HF_TOKEN"))
        print(f"{name}: {paths[name]} ({os.path.getsize(paths[name]) / 1e9:.2f} GB)", flush=True)

    manifest_path = os.path.join(args.out, "manifest.json")
    manifest = json.load(open(manifest_path)) if os.path.exists(manifest_path) else {"sources": {}}
    seen = set()
    report = {}
    val_part = os.path.join(args.out, "parts", "val_sc.bin")
    for name in args.sources:
        part = os.path.join(args.out, "parts", f"{name}.bin")
        done = part + ".done"
        if os.path.exists(done) and not args.dry_stats:
            print(f"[{name}] already built, skipping", flush=True)
            continue
        stats = Counter()
        t0 = time.time()

        def cleaned():
            for raw in iter_texts(paths[name]):
                stats["docs_read"] += 1
                if args.max_docs and stats["docs_read"] > args.max_docs:
                    return
                text = " ".join(raw.split())
                scr = dominant_script(text)
                stats[f"script_read_{scr}"] += 1
                scripts.append(scr)
                if text.count(" ") + 1 < args.min_words:
                    stats["dropped_short"] += 1
                    continue
                h = doc_hash(text)
                if h in seen:
                    stats["dropped_duplicate"] += 1
                    continue
                seen.add(h)
                if name == "supreme_court" and h % 100 < args.val_percent:
                    stats["held_out_val"] += 1
                    yield ("val", text)
                    continue
                yield ("train", text)

        texts, kinds, scripts, kept_scripts = [], [], [], []

        def feed():
            for kind, text in cleaned():
                kinds.append(kind)
                kept_scripts.append(scripts[-1])          # script of the document being yielded
                yield text

        mode = "wb" if name == "supreme_court" else "ab"     # val.bin holds SC docs only; a rebuild starts it fresh
        out_path, val_path_ = (os.devnull, os.devnull) if args.dry_stats else (part, val_part)
        with open(out_path, "wb") as out, open(val_path_, mode) as vout, tok.pool(args.workers) as pool:
            for i, (arr, n_words) in enumerate(pool.imap(a1_tokenizer.encode_array_worker, feed(), chunksize=16)):
                kind, scr = kinds[i], kept_scripts[i]
                if (not args.no_fertility_filter and kind == "train" and name != "supreme_court"
                        and len(arr) / max(1, n_words) > args.max_fertility):
                    stats["dropped_fertility"] += 1
                    stats[f"script_dropped_fertility_{scr}"] += 1
                    stats[f"tokens_dropped_fertility_{scr}"] += len(arr)
                    continue
                if args.dry_stats:
                    stats[f"script_kept_{scr}"] += 1
                    stats[f"tokens_kept_{scr}"] += len(arr) + 1
                    stats["docs_kept"] += kind == "train"
                    continue
                stats[f"script_kept_{scr}"] += 1
                stats[f"tokens_kept_{scr}"] += len(arr) + 1
                arr = np.append(arr, np.uint16(EOT)).astype(np.uint16)
                (vout if kind == "val" else out).write(arr.tobytes())
                stats["val_tokens" if kind == "val" else "train_tokens"] += len(arr)
                stats["docs_kept"] += kind == "train"
                if stats["docs_kept"] and stats["docs_kept"] % 100_000 == 0 and kind == "train":
                    el = time.time() - t0
                    print(f"  [{name}] {stats['docs_kept']:,} docs | {stats['train_tokens'] / 1e6:,.0f}M tokens | "
                          f"{stats['train_tokens'] / el / 1e6:.2f}M tok/s", flush=True)
        stats["minutes"] = round((time.time() - t0) / 60, 1)
        if args.dry_stats:
            report[name] = dict(stats)
            print(f"[{name}] (dry) {dict(stats)}", flush=True)
            continue
        manifest["sources"][name] = dict(stats)
        json.dump(manifest, open(manifest_path, "w"), indent=2)
        open(done, "w").write("ok")
        print(f"[{name}] {dict(stats)}", flush=True)
    if args.dry_stats:
        json.dump(report, open(os.path.join(args.out, "script_stats.json"), "w"), indent=2)
        print("dry run: per-script counts written to", os.path.join(args.out, "script_stats.json"))
        return

    # final train.bin / val.bin
    train_path, val_path = os.path.join(args.out, "train.bin"), os.path.join(args.out, "val.bin")
    with open(train_path, "wb") as out:
        for name in args.sources:
            with open(os.path.join(args.out, "parts", f"{name}.bin"), "rb") as f:
                shutil.copyfileobj(f, out, length=64 << 20)
    shutil.copyfile(val_part, val_path)
    n_train = os.path.getsize(train_path) // 2
    n_val = os.path.getsize(val_path) // 2
    probe = np.memmap(train_path, dtype=np.uint16, mode="r")
    assert int(probe[: min(len(probe), 50_000_000)].max()) < V, "token id >= vocab: tokenizer mismatch"
    manifest.update(tokenizer=os.path.basename(args.tokenizer), tokenizer_sha256=tok_sha, vocab_size=V, eot_id=EOT,
                    dtype="uint16", train_tokens=n_train, val_tokens=n_val, whitespace_normalised=True,
                    min_words=args.min_words, max_fertility=None if args.no_fertility_filter else args.max_fertility,
                    fertility_filter=not args.no_fertility_filter, val_percent=args.val_percent,
                    build_minutes_total=round((time.time() - t_all) / 60, 1))
    json.dump(manifest, open(manifest_path, "w"), indent=2)
    print(f"train.bin {n_train:,} tokens ({n_train * 2 / 1e9:.1f} GB) | val.bin {n_val:,} tokens")
    print("parts/ can be deleted to free disk once you are happy with train.bin")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile train_300.py
# Plumbing: pretrain one decoder-only model (your arch3 code) on every visible GPU (adapted from the Phase-1 train_ddp.py)
"""Pretrain one hand-written decoder (arch3) from scratch on all visible GPUs (written with Claude; declare it).

Changes vs. the Phase-1 Kaggle version (train_ddp.py):
  * bf16 autocast when the GPU supports it (L40S / A10G / A100), no GradScaler; fp16 + GradScaler otherwise;
  * schedule normally driven by train.max_tokens (deadline_unix is just a safety stop for cloud runs);
  * SIGTERM / SIGINT (instance stop, spot interruption, Ctrl-C) -> finish the step, evaluate, save last.pt, exit;
  * resume_from = <out_dir>/last.pt continues exactly (optimizer, step, tokens, schedule);
  * generation is skipped when gen.decoding is empty (pretraining only).
best.pt / last.pt keep the Phase-1 format {"model", "config": {"module", "model", ...}}, so the Phase-2+ pipeline loads
them like B2.

Usage:  torchrun --standalone --nproc_per_node=2 train_ddp.py run_B.json
        (also runs as a single process: python train_ddp.py run_B.json)

Same model / optimiser / schedule semantics as the single-GPU train_lm.py of the A/B run:
  * the global batch (batch_tokens) is split evenly over the GPUs; each GPU runs micro-batches of its share,
    gradients are averaged across GPUs (DDP), so one optimiser step still sees batch_tokens tokens;
  * the LR schedule is driven by wall-clock progress to `deadline_unix` (or max_tokens), computed on rank 0
    and broadcast every step, so all GPUs always use the same LR and stop at the same step;
  * eval, checkpoints, logging and generation happen on rank 0 only; checkpoints hold the plain (unwrapped)
    state_dict, loadable exactly like the A/B checkpoints (gen_local.py works unchanged).
Validation uses a fixed set of windows (eval_windows, default 360 = the A/B/C/D runs' 40 x 9), so val losses
are comparable with the earlier runs.

Probe mode (cfg["probe"]): same code path on synthetic tokens for a fixed number of minutes, then times
worst-case generation per decoding strategy instead of writing predictions.
"""
import os
import sys
import json
import math
import time
import queue
import random
import threading
import traceback
import gc
import importlib
from contextlib import nullcontext
from datetime import timedelta

import numpy as np
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less fragmentation on long runs
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

import fastattn
from a1_tokenizer import ByteBPE

# ---------------------------------------------------------------------------
# distributed setup
# ---------------------------------------------------------------------------
WORLD = int(os.environ.get("WORLD_SIZE", "1"))
RANK = int(os.environ.get("RANK", "0"))
LOCAL_RANK = int(os.environ.get("LOCAL_RANK", "0"))
DIST = WORLD > 1
CUDA = torch.cuda.is_available()         # CPU path exists only for local smoke tests
if CUDA:
    torch.cuda.set_device(LOCAL_RANK)
device = torch.device(f"cuda:{LOCAL_RANK}" if CUDA else "cpu")
if DIST:
    dist.init_process_group("nccl" if CUDA else "gloo", timeout=timedelta(minutes=10))
MASTER = RANK == 0

cfg = json.load(open(sys.argv[1]))
TAG, OUT = cfg["tag"], cfg["out_dir"]
os.makedirs(OUT, exist_ok=True)
TR, GEN = cfg["train"], cfg["gen"]
DEADLINE = cfg["deadline_unix"]
SESSION_END = cfg["session_end_unix"]


def log(msg, force=False, **kv):
    """rank 0 logs everything; other ranks only what is passed with force=True (errors)."""
    if not (MASTER or force):
        return
    rec = {"t": round(time.time(), 1), "msg": msg, **kv}
    if not MASTER:
        rec["rank"] = RANK
    print(json.dumps(rec), flush=True)
    name = "log.jsonl" if MASTER else f"log_rank{RANK}.jsonl"
    with open(os.path.join(OUT, name), "a") as f:
        f.write(json.dumps(rec) + "\n")


seed = TR["seed"]
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)     # identical init on every rank
BF16 = CUDA and torch.cuda.is_bf16_supported()
AMP = torch.bfloat16 if BF16 else torch.float16     # bf16 on L40S/A10G/A100; fp16 + GradScaler on T4/V100
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ---------------------------------------------------------------------------
# model
# ---------------------------------------------------------------------------
arch = importlib.import_module(cfg["module"])
mcfg = arch.LMConfig(**cfg["model"])
raw_model = arch.DecoderLM(mcfg)
if cfg.get("init_std"):
    std = cfg["init_std"]
    for m in raw_model.modules():
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, std)
            if getattr(m, "bias", None) is not None:
                nn.init.zeros_(m.bias)
n_params = sum(p.numel() for p in raw_model.parameters())
if TR.get("fast_attention", True):
    fastattn.patch_model(raw_model)
raw_model.to(device)
init_from = cfg.get("init_from")
if init_from and not (cfg.get("resume_from") and os.path.exists(cfg["resume_from"])):
    # warm start: weights only (fresh optimiser, fresh warmup + schedule); the architecture must match exactly
    ck = torch.load(init_from, map_location="cpu", weights_only=False)
    src = ck.get("config", {})
    assert src.get("module") == cfg["module"] and src.get("model") == cfg["model"], \
        f"init_from checkpoint architecture {src.get('module')} {src.get('model')} != this run's {cfg['module']} {cfg['model']}"
    raw_model.load_state_dict(ck["model"])
    log("init_from", path=init_from, src_step=ck.get("step"), src_tokens_M=round(ck.get("tokens", 0) / 1e6, 1),
        src_best_val=ck.get("best_val"))
    del ck
T = mcfg.max_seq_len
V = mcfg.vocab_size
assert GEN["max_new_tokens"] < T, "gen.max_new_tokens must be < max_seq_len"
log("model", tag=TAG, module=cfg["module"], params_M=round(n_params / 1e6, 2), seq_len=T, world_size=WORLD,
    precision="bf16" if BF16 else ("fp16" if CUDA else "fp32"), gpu=torch.cuda.get_device_name(0) if CUDA else "cpu")

# ---------------------------------------------------------------------------
# data: random windows from the uint16 token stream, prefetched on a thread (different stream per rank)
# ---------------------------------------------------------------------------
train_mm = np.memmap(os.path.join(cfg["data_dir"], "train.bin"), dtype=np.uint16, mode="r")
val_mm = np.memmap(os.path.join(cfg["data_dir"], "val.bin"), dtype=np.uint16, mode="r")
assert int(train_mm[:10_000_000].max()) < V, "token id >= vocab_size: data/tokenizer mismatch"


def windows(mm, starts):
    arr = np.stack([mm[s:s + T + 1] for s in starts]).astype(np.int64)
    t = torch.from_numpy(arr)
    t = t.pin_memory() if CUDA else t
    return t[:, :-1], t[:, 1:]


class Prefetcher:
    def __init__(self, mb, seed):
        self.mb, self.rng = mb, np.random.default_rng(seed)
        self.q = queue.Queue(maxsize=6)
        threading.Thread(target=self._run, daemon=True).start()

    def _run(self):
        hi = len(train_mm) - T - 1
        while True:
            self.q.put(windows(train_mm, self.rng.integers(0, hi, self.mb)))

    def next(self):
        x, y = self.q.get()
        return x.to(device, non_blocking=True), y.to(device, non_blocking=True)


def loss_fn(model, x, y):
    with torch.autocast("cuda", dtype=AMP, enabled=CUDA):
        logits = model(x)
        return F.cross_entropy(logits.view(-1, logits.size(-1)), y.reshape(-1))


def all_min(v):
    if not DIST:
        return v
    t = torch.tensor([v], device=device, dtype=torch.long)
    dist.all_reduce(t, op=dist.ReduceOp.MIN)
    return int(t.item())


# ---------------------------------------------------------------------------
# batch: global batch split over the GPUs; micro-batch = largest divisor of the per-GPU share that fits
# ---------------------------------------------------------------------------
batch_seqs = max(1, TR["batch_tokens"] // T)
assert batch_seqs % WORLD == 0, (f"batch_tokens // max_seq_len = {batch_seqs} sequences must be divisible by the "
                                 f"{WORLD} GPUs (e.g. batch_tokens = {WORLD * math.ceil(batch_seqs / WORLD) * T})")
rank_seqs = batch_seqs // WORLD
free_mem = torch.cuda.mem_get_info()[0] + torch.cuda.memory_allocated() if CUDA else 0
opt_state_bytes = 2 * 4 * sum(p.numel() for p in raw_model.parameters() if p.requires_grad)
ddp_bytes = 4 * n_params if DIST else 0                      # DDP gradient buckets
budget = free_mem - opt_state_bytes - ddp_bytes - max(1.5e9, 0.08 * free_mem)    # headroom for resume / fragmentation
candidates = sorted([d for d in range(1, min(rank_seqs, TR.get("max_micro_batch", 64)) + 1)
                     if rank_seqs % d == 0], reverse=True)
mb, peak = None, 0
if not CUDA:
    mb = min(candidates, key=lambda d: abs(d - 2))
for cand in candidates if CUDA else []:
    try:
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        x, y = windows(train_mm, np.arange(cand) * 997)
        raw_model.train()
        loss_fn(raw_model, x.to(device), y.to(device)).backward()
        raw_model.zero_grad(set_to_none=True)
        peak = torch.cuda.max_memory_allocated()
        if peak < budget:
            mb = cand
            break
    except torch.cuda.OutOfMemoryError:
        pass
    raw_model.zero_grad(set_to_none=True)
    gc.collect()
    torch.cuda.empty_cache()
mb = all_min(mb if mb is not None else 0)                     # same micro-batch on every GPU
assert mb >= 1, "even micro-batch 1 does not fit; reduce max_seq_len or model size"
accum = rank_seqs // mb
log("batch", global_seqs=batch_seqs, gpus=WORLD, seqs_per_gpu=rank_seqs, micro_batch=mb, grad_accum=accum,
    tokens_per_step=batch_seqs * T, probe_peak_GB=round(peak / 1e9, 2))

ddp = DDP(raw_model, device_ids=[LOCAL_RANK] if CUDA else None, broadcast_buffers=False,
          gradient_as_bucket_view=True) if DIST else None
base = ddp if DIST else raw_model                              # what the training step calls
model = base
if TR.get("compile", False):
    t_compile = time.time()
    try:
        model = torch.compile(base)
        x, y = windows(train_mm, np.arange(mb) * 991)
        loss_fn(model, x.to(device), y.to(device)).backward()
        raw_model.zero_grad(set_to_none=True)
        log("torch.compile ok", seconds=round(time.time() - t_compile, 1))
    except Exception as e:                                     # fall back to eager
        model = base
        raw_model.zero_grad(set_to_none=True)
        log("torch.compile failed, using eager", error=repr(e)[:300])

# ---------------------------------------------------------------------------
# optimiser (weight decay on >=2-D weights only); one per rank, identical updates
# ---------------------------------------------------------------------------
decay = [p for p in raw_model.parameters() if p.requires_grad and p.dim() >= 2]
no_decay = [p for p in raw_model.parameters() if p.requires_grad and p.dim() < 2]
groups = [{"params": decay, "weight_decay": TR["weight_decay"]}, {"params": no_decay, "weight_decay": 0.0}]
try:
    opt = torch.optim.AdamW(groups, lr=TR["lr"], betas=tuple(TR["betas"]), eps=TR.get("eps", 1e-8), fused=CUDA)
except (RuntimeError, TypeError):
    opt = torch.optim.AdamW(groups, lr=TR["lr"], betas=tuple(TR["betas"]), eps=TR.get("eps", 1e-8))
scaler = torch.amp.GradScaler("cuda", enabled=CUDA and not BF16)   # bf16 needs no loss scaling

step, tokens_seen, train_seconds_before, best_val = 0, 0, 0.0, float("inf")
warmup_end_progress = None
if cfg.get("resume_from") and os.path.exists(cfg["resume_from"]):
    ck = torch.load(cfg["resume_from"], map_location="cpu", weights_only=False)   # CPU first: no 2x GPU memory spike on resume
    raw_model.load_state_dict(ck["model"])
    opt.load_state_dict(ck["opt"])
    scaler.load_state_dict(ck["scaler"])
    step, tokens_seen = ck["step"], ck["tokens"]
    train_seconds_before, best_val = ck["train_seconds"], ck.get("best_val", float("inf"))
    warmup_end_progress = ck.get("warmup_end_progress")
    del ck
    gc.collect()
    log("resumed", path=cfg["resume_from"], step=step, tokens=tokens_seen)

# ---------------------------------------------------------------------------
# schedule: warmup in steps, then decay over wall-clock progress (computed on rank 0, broadcast)
# ---------------------------------------------------------------------------
if cfg.get("probe"):                     # runtime probe: fixed training window from here
    DEADLINE = min(DEADLINE, time.time() + cfg["probe"]["train_minutes"] * 60)
t_start = time.time()
schedule_seconds = (cfg["schedule_hours"] * 3600 if cfg.get("schedule_hours")
                    else train_seconds_before + (DEADLINE - t_start))
max_tokens = TR.get("max_tokens")


def progress_now():
    p = (train_seconds_before + time.time() - t_start) / schedule_seconds
    if max_tokens:
        p = max(p, tokens_seen / max_tokens)
    return p


def lr_at(step, p):
    global warmup_end_progress
    lr, min_lr, warm = TR["lr"], TR["min_lr"], TR["warmup_steps"]
    if step < warm:
        return lr * (step + 1) / warm
    if warmup_end_progress is None:
        warmup_end_progress = min(p, 0.99)
    q = min(1.0, max(0.0, (p - warmup_end_progress) / (1 - warmup_end_progress)))
    kind = TR.get("schedule", "cosine")
    if kind == "cosine":
        return min_lr + 0.5 * (lr - min_lr) * (1 + math.cos(math.pi * q))
    if kind == "linear":
        return lr - (lr - min_lr) * q
    return lr


def control():
    """rank 0 decides (lr, progress, stop); every rank receives the same values."""
    t = torch.zeros(3, dtype=torch.float64, device=device)
    if MASTER:
        p = progress_now()
        stop = time.time() >= DEADLINE or p >= 1.0 or STOP_REQUESTED["flag"]
        t[0], t[1], t[2] = (0.0 if stop else lr_at(step, p)), p, float(stop)
    if DIST:
        dist.broadcast(t, src=0)
    return t[0].item(), t[1].item(), bool(t[2].item())


# ---------------------------------------------------------------------------
# eval + checkpoints (rank 0 only, no collectives)
# ---------------------------------------------------------------------------
val_starts = np.random.default_rng(1234).integers(0, len(val_mm) - T - 1, TR.get("eval_windows", 360))


@torch.no_grad()
def evaluate():
    raw_model.eval()
    tot, n = 0.0, 0
    for i in range(0, len(val_starts), mb):
        x, y = windows(val_mm, val_starts[i:i + mb])
        tot += loss_fn(raw_model, x.to(device), y.to(device)).item() * len(x)
        n += len(x)
    raw_model.train()
    return tot / n


def save(path, full):
    state = {"model": raw_model.state_dict(), "config": cfg, "step": step, "tokens": tokens_seen,
             "train_seconds": train_seconds_before + time.time() - t_start, "best_val": best_val,
             "warmup_end_progress": warmup_end_progress, "world_size": WORLD}
    if full:
        state.update(opt=opt.state_dict(), scaler=scaler.state_dict())
    torch.save(state, path + ".tmp")
    os.replace(path + ".tmp", path)


def do_eval(tag):
    global best_val
    t0 = time.time()
    vl = evaluate()
    t_eval = time.time() - t0
    improved = vl < best_val
    if improved:
        best_val = vl
        save(os.path.join(OUT, "best.pt"), full=False)
    save(os.path.join(OUT, "last.pt"), full=True)
    log(tag, step=step, tokens=tokens_seen, val_loss=round(vl, 4), val_ppl=round(math.exp(vl), 2),
        best=improved, hours=round((time.time() - t_start) / 3600, 2),
        eval_s=round(t_eval, 1), save_s=round(time.time() - t0 - t_eval, 1))


# ---------------------------------------------------------------------------
# train
# ---------------------------------------------------------------------------
import signal
STOP_REQUESTED = {"flag": False}


def _request_stop(signum, frame):
    STOP_REQUESTED["flag"] = True
    log("stop signal received: finishing this step, then evaluate + save last.pt", force=True, signal=signum)


signal.signal(signal.SIGTERM, _request_stop)
signal.signal(signal.SIGINT, _request_stop)
pf = Prefetcher(mb, seed + 1000 * RANK + step)
raw_model.train()
last_eval = time.time()
last_log, tok_at_last_log = time.time(), tokens_seen
eval_every = TR["eval_every_min"] * 60
bad_logs = 0
log("training", deadline_in_h=round((DEADLINE - t_start) / 3600, 2), schedule_hours=round(schedule_seconds / 3600, 2))


def train_step(lr):
    for g in opt.param_groups:
        g["lr"] = lr
    loss_sum = torch.zeros((), device=device)
    for i in range(accum):
        x, y = pf.next()
        sync = (i == accum - 1) or not DIST
        with (nullcontext() if sync else ddp.no_sync()):       # all-reduce once per optimiser step
            loss = loss_fn(model, x, y) / accum
            scaler.scale(loss).backward()
        loss_sum += loss.detach()
    if TR.get("grad_clip"):
        scaler.unscale_(opt)
        gnorm = torch.nn.utils.clip_grad_norm_(raw_model.parameters(), TR["grad_clip"]).item()
    else:
        gnorm = float("nan")
    scaler.step(opt)
    scaler.update()
    opt.zero_grad(set_to_none=True)
    return loss_sum, gnorm


while True:
    try:
        lr, p, stop = control()
        if stop:
            break
        loss_sum, gnorm = train_step(lr)
    except Exception:
        # with several GPUs a failure on one rank cannot be recovered in step; stop cleanly instead
        opt.zero_grad(set_to_none=True)
        log("training error; stopping training (rank 0 then evaluates, saves and generates)", force=True,
            step=step, error=traceback.format_exc()[-1500:])
        break
    step += 1
    tokens_seen += batch_seqs * T

    if step % TR.get("log_every", 50) == 0:
        if DIST:
            dist.all_reduce(loss_sum, op=dist.ReduceOp.AVG if CUDA else dist.ReduceOp.SUM)
            if not CUDA:
                loss_sum /= WORLD
        loss_val = loss_sum.item()
        now = time.time()
        log("train", step=step, loss=round(loss_val, 4), lr=lr, grad_norm=round(gnorm, 3),
            tok_per_s=int((tokens_seen - tok_at_last_log) / (now - last_log)),
            tokens_M=round(tokens_seen / 1e6, 1), progress=round(p, 4),
            mem_GB=round(torch.cuda.max_memory_allocated() / 1e9, 2) if CUDA else 0.0)
        last_log, tok_at_last_log = now, tokens_seen
        bad_logs = 0 if math.isfinite(loss_val) else bad_logs + 1
        if bad_logs >= 3:
            log("loss non-finite for 3 log intervals; stopping training (generation uses the best checkpoint)",
                step=step)
            break                                              # every rank sees the same averaged loss
    if MASTER and time.time() - last_eval >= eval_every:
        do_eval("eval")
        last_eval = time.time()

del pf
if DIST and os.environ.get("DDP_CHECK"):                       # smoke tests: every rank must hold identical weights
    with torch.no_grad():
        s = torch.stack([p.double().sum() for p in raw_model.parameters()]).sum().view(1)
    sums = [torch.zeros_like(s) for _ in range(WORLD)]
    dist.all_gather(sums, s)
    log("ddp_check", param_sums=[round(x.item(), 6) for x in sums], identical=len({x.item() for x in sums}) == 1)
if not MASTER:                                                 # only rank 0 continues
    if DIST:
        try:
            dist.destroy_process_group()
        except Exception:
            pass
    sys.exit(0)

try:
    do_eval("final_eval")
except Exception:
    log("final eval failed", error=traceback.format_exc()[-1500:])
    save(os.path.join(OUT, "last.pt"), full=False)
log("training done", steps=step, tokens=tokens_seen, best_val=round(best_val, 4))
if DIST:
    try:
        dist.destroy_process_group()
    except Exception:
        pass

# ---------------------------------------------------------------------------
# generation (rank 0): one prediction file per decoding strategy (+ held-out val prompts for local scoring)
# ---------------------------------------------------------------------------
if not GEN.get("decoding"):                                    # training-only run: generate elsewhere from best.pt
    log("training only (no decoding strategies): best.pt / last.pt are the outputs")
    log("all done")
    sys.exit(0)

del opt
if CUDA:
    torch.cuda.empty_cache()
which = GEN.get("checkpoint", "best")
if not os.path.exists(os.path.join(OUT, f"{which}.pt")):
    log(f"{which}.pt missing (no finite val loss?); falling back to last.pt")
    which = "last"
ck = torch.load(os.path.join(OUT, f"{which}.pt"), map_location=device, weights_only=False)
raw_model.load_state_dict(ck["model"])
raw_model.eval()
if not TR.get("fast_attention", True):
    fastattn.patch_model(raw_model)          # KV cache needs the patched forward


class ProbeTokenizer:
    """Stand-in for the real tokenizer in probe mode: every prompt encodes to `prompt_len` random ids
    and no token ever ends a word, so each batch runs the full max_new_tokens (worst case)."""
    n_special, eow_id = 32, 0

    def __init__(self, vocab_size, prompt_len):
        self.token_eow = [False] * vocab_size
        self.token_bytes = [b"x"] * vocab_size
        self._ids = np.random.default_rng(0).integers(0, vocab_size, prompt_len).tolist()

    def encode(self, text):
        return self._ids

    def decode(self, ids):
        return ""


tokenizer = ProbeTokenizer(V, T - GEN["max_new_tokens"]) if cfg.get("probe") else ByteBPE.from_json(cfg["tokenizer_path"])
STRATEGIES = list(GEN["decoding"].items())         # [(name, {temperature, top_k, top_p, repetition_penalty}), ...]


def run_gen(prompts, dec):
    bs = GEN["batch_size"]
    while True:
        try:
            return fastattn.generate_words(
                raw_model, tokenizer, prompts, max_seq_len=T, target_words=GEN["target_words"],
                max_new_tokens=GEN["max_new_tokens"], batch_size=bs, temperature=dec["temperature"],
                top_k=dec.get("top_k"), top_p=dec.get("top_p"), repetition_penalty=dec.get("repetition_penalty", 1.0),
                device=device, amp_dtype=AMP if CUDA else torch.float32,
                no_repeat_ngram=dec.get("no_repeat_ngram", 0), ngram_scope=dec.get("ngram_scope", "output"))
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            bs //= 2
            assert bs >= 1
            log("generation OOM, retrying", batch_size=bs)


def looping_share(texts):
    """share of predictions in which some 4-word sequence occurs 3+ times (a simple repetition indicator)."""
    n = 0
    for t in texts:
        w = t.split()
        grams = {}
        for i in range(len(w) - 3):
            g = tuple(w[i:i + 4])
            grams[g] = grams.get(g, 0) + 1
        n += bool(grams) and max(grams.values()) >= 3
    return round(n / max(1, len(texts)), 3)


if cfg.get("probe"):
    for name, dec in STRATEGIES:
        times = []
        for _ in range(2):                   # the first batch includes CUDA warm-up
            if CUDA:
                torch.cuda.synchronize()
            t0 = time.time()
            run_gen(["probe"] * GEN["batch_size"], dec)
            if CUDA:
                torch.cuda.synchronize()
            times.append(time.time() - t0)
        log("probe_generation", strategy=name, batch_size=GEN["batch_size"], prompt_tokens=T - GEN["max_new_tokens"],
            new_tokens=GEN["max_new_tokens"], first_batch_s=round(times[0], 2), batch_s=round(times[1], 2))
    log("all done")
    sys.exit(0)

test = json.load(open(cfg["test_path"], encoding="utf-8"))
ids = list(test.keys())
test_prompts = [test[k]["text"] for k in ids]
last_minutes = 0.0
for i, (name, dec) in enumerate(STRATEGIES):
    if i and SESSION_END - time.time() < 1.5 * last_minutes * 60 + 300:
        log("skipping decoding strategy: not enough session time left", strategy=name)
        continue
    torch.manual_seed(GEN.get("seed", 0))
    t0 = time.time()
    preds = run_gen(test_prompts, dec)
    last_minutes = (time.time() - t0) / 60
    prediction = {k: {"text": p} for k, p in zip(ids, preds)}
    paths = [os.path.join(OUT, f"prediction_{name}.json")] + ([os.path.join(OUT, "prediction.json")] if i == 0 else [])
    for path in paths:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(prediction, f, ensure_ascii=False, indent=2)
    short = sum(len(p.split()) < GEN["target_words"] for p in preds)
    log("prediction.json written", strategy=name, decoding=dec, n=len(preds), minutes=round(last_minutes, 1),
        under_target_words=short, looping_share=looping_share(preds), checkpoint=which)

vp_path = os.path.join(cfg["data_dir"], "val_prompts.json")
n_val = GEN.get("n_val_prompts", 0)
if n_val and os.path.exists(vp_path):
    vp = json.load(open(vp_path, encoding="utf-8"))
    keys = list(vp)[:n_val]
    for name, dec in STRATEGIES:
        if SESSION_END - time.time() < 600 + 0.25 * last_minutes * 60:
            log("skipping val generations: not enough session time left", strategy=name)
            break
        try:
            torch.manual_seed(GEN.get("seed", 0))
            vpreds = run_gen([vp[k]["text"] for k in keys], dec)
            with open(os.path.join(OUT, f"val_generations_{name}.json"), "w", encoding="utf-8") as f:
                json.dump({k: {"prediction": p, "reference": vp[k]["reference"]} for k, p in zip(keys, vpreds)},
                          f, ensure_ascii=False, indent=2)
            log("val_generations written", strategy=name, n=len(keys), looping_share=looping_share(vpreds))
        except Exception:
            log("val generation failed (prediction files are unaffected)", strategy=name,
                error=traceback.format_exc()[-1500:])
log("all done")

## 3 — parameter count

In [ ]:
# parameter count (CPU, no weights needed) + tokens per parameter
import importlib, arch3
importlib.reload(arch3)
m = arch3.DecoderLM(arch3.LMConfig(**MODEL))
n = sum(p.numel() for p in m.parameters())
emb = MODEL["vocab_size"] * MODEL["d_model"]
print(f"parameters: {n:,} ({n / 1e6:.1f}M) | embedding (tied with the LM head) {emb / 1e6:.1f}M | blocks {(n - emb) / 1e6:.1f}M")
_mt = TRAIN["max_tokens"] if TRAIN["max_tokens"] != "all" else (
    json.load(open(os.path.join(DATA_DIR, "manifest.json")))["train_tokens"] if os.path.exists(os.path.join(DATA_DIR, "manifest.json")) else 4.5e9)
print(f"tokens per parameter at max_tokens ({_mt / 1e9:.2f}B): {_mt / n:.1f}   (B2: ~8.9; Chinchilla-optimal ~20)")
assert n <= 300_000_000, "over the 300M Phase-1 limit"
if n < 250_000_000:
    print("note: below your 250-300M target")
del m

## 4a — optional: language / script mix (counts only)

In [ ]:
# OPTIONAL, before the full build: language / script mix of the corpus (counts only, writes no token files).
# Per source: documents read per dominant script, and which would be KEPT vs DROPPED by the fertility filter
# (non-Supreme-Court docs with > 2.0 tokens/word are dropped; this removes most non-English text, since your tokenizer
# needs ~4 tokens/word for Hindi and ~20-27 for Tamil/Bengali). Supreme Court docs are exempt from the filter.
# Whether to change the filter (build_data.py --max-fertility) is your decision.
subprocess.run([sys.executable, "build_data.py", "--tokenizer", TOKENIZER, "--out", os.path.join(WORK, "lang_stats"),
                "--dry-stats"], check=True)
rep = json.load(open(os.path.join(WORK, "lang_stats", "script_stats.json")))
for src, st in rep.items():
    scripts = sorted({k.split("_", 2)[2] for k in st if k.startswith("script_read_")})
    print(f"\n{src}: {st.get('docs_read', 0):,} docs read")
    for sc in scripts:
        kept_t, drop_t = st.get(f"tokens_kept_{sc}", 0), st.get(f"tokens_dropped_fertility_{sc}", 0)
        print(f"  {sc:12s} docs read {st.get(f'script_read_{sc}', 0):>9,} | kept {st.get(f'script_kept_{sc}', 0):>9,} "
              f"({kept_t / 1e6:,.1f}M tok) | dropped by fertility {st.get(f'script_dropped_fertility_{sc}', 0):>8,} ({drop_t / 1e6:,.1f}M tok)")

## 4 — data (whole corpus, once)

In [ ]:
# build train.bin / val.bin from the WHOLE corpus with your tokenizer (once; resumable; ~1-2 h on 8 vCPUs, mostly encoding)
if os.path.exists(os.path.join(DATA_DIR, "train.bin")) and os.path.exists(os.path.join(DATA_DIR, "manifest.json")):
    print("data already built:"); print(open(os.path.join(DATA_DIR, "manifest.json")).read()[:3000])
else:
    assert os.environ.get("HF_TOKEN"), "export HF_TOKEN before starting Jupyter (gated dataset)"
    subprocess.run([sys.executable, "build_data.py", "--tokenizer", TOKENIZER, "--out", DATA_DIR]
                   + ([] if FERTILITY_FILTER else ["--no-fertility-filter"]), check=True)
man = json.load(open(os.path.join(DATA_DIR, "manifest.json")))
print(f"fertility filter: {man.get('fertility_filter')} | min words {man.get('min_words')} | exact-duplicate removal on")
print(f"train tokens {man['train_tokens']:,} | val tokens {man['val_tokens']:,} | max_tokens / train tokens = "
      f"{resolved_train()['max_tokens'] / man['train_tokens']:.2f} passes")

## 5 — runtime probe (~10 min)

In [ ]:
# runtime probe: the real training script on the real data for PROBE_MIN minutes (separate folder), then a projection
PROBE_MIN = 8
pdir = os.path.join(WORK, "probe")
shutil.rmtree(pdir, ignore_errors=True)
subprocess.run([sys.executable, "train_300.py", run_config(pdir, probe_minutes=PROBE_MIN)], check=True)
recs = [json.loads(l) for l in open(os.path.join(pdir, "log.jsonl"))]
b = [r for r in recs if r["msg"] == "batch"][0]
tr = [r for r in recs if r["msg"] == "train"]
tok_s = sorted(r["tok_per_s"] for r in tr[len(tr) // 3:])[len(tr[len(tr) // 3:]) // 2]     # median after warm-up
mem = max(r["mem_GB"] for r in tr)
MT = resolved_train()["max_tokens"]
hours = MT / tok_s / 3600
print(f"micro-batch {b['micro_batch']} x accum {b['grad_accum']} | {tok_s:,} tokens/s | peak memory {mem} GB")
print(f"projected training time for {MT / 1e9:.2f}B tokens: {hours:.1f} h"
      + (f" | ~${hours * COST_PER_HOUR:,.0f} at ${COST_PER_HOUR}/h" if COST_PER_HOUR else ""))
compiled = [r for r in recs if "compile" in r["msg"]]
print("torch.compile:", compiled[0]["msg"] if compiled else "not used")

## 6 — launch training (background)

In [ ]:
# launch the real run in the BACKGROUND (keeps training if the browser/notebook disconnects; auto-resumes from last.pt)
def is_running(pid_file):
    # True only for a live training process (finished children are reaped, zombies count as stopped)
    if not os.path.exists(pid_file):
        return False
    pid = int(open(pid_file).read().strip())
    try:
        os.waitpid(pid, os.WNOHANG)          # reap it if it is our finished child
    except ChildProcessError:
        pass
    try:
        return open(f"/proc/{pid}/stat").read().split()[2] != "Z"
    except FileNotFoundError:
        return False

cfg_path = run_config(OUT_DIR)
log_path = os.path.join(OUT_DIR, "train.stdout")
pid_path = os.path.join(OUT_DIR, "pid")
if is_running(pid_path):
    print("already running, pid", open(pid_path).read().strip())
else:
    proc = subprocess.Popen(["nohup", sys.executable, "train_300.py", cfg_path], stdout=open(log_path, "a"),
                            stderr=subprocess.STDOUT, cwd=WORK, start_new_session=True)
    open(pid_path, "w").write(str(proc.pid))
    print("started pid", proc.pid, "| log", log_path)
print("stop gracefully (evaluates + saves last.pt):  kill -TERM $(cat run_B3/pid)   |  resume: run this cell again")

## 7 — monitor

In [ ]:
# monitor (run any time): last log lines + loss curves
import matplotlib.pyplot as plt
lj = os.path.join(OUT_DIR, "log.jsonl")
recs = [json.loads(l) for l in open(lj)] if os.path.exists(lj) else []
for r in recs[-6:]:
    print({k: v for k, v in r.items() if k != "t"})
tr = [r for r in recs if r["msg"] == "train"]; ev = [r for r in recs if r["msg"] in ("eval", "final_eval")]
if tr:
    MT = resolved_train()["max_tokens"]
    done = tr[-1]["tokens_M"] * 1e6 / MT
    print(f"progress {100 * done:.1f}% | {tr[-1]['tok_per_s']:,} tok/s | peak mem {tr[-1]['mem_GB']} GB | "
          f"ETA {(1 - done) * MT / max(1, tr[-1]['tok_per_s']) / 3600:.1f} h")
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot([r["tokens_M"] for r in tr], [r["loss"] for r in tr]); ax[0].set_title("train loss"); ax[0].set_xlabel("tokens (M)")
    if ev:
        ax[1].plot([r["tokens"] / 1e6 for r in ev], [r["val_loss"] for r in ev], "o-"); ax[1].set_title("val loss")
    plt.show()
pid_path = os.path.join(OUT_DIR, "pid")
print("training process running:", is_running(pid_path))

## 8 — export for Phase 3+

In [ ]:
# export: best.pt + tokenizer + manifest into export/ (upload this folder as a PRIVATE Kaggle dataset for Phase 3+)
exp = os.path.join(WORK, "export", "run_B3")
os.makedirs(exp, exist_ok=True)
best = os.path.join(OUT_DIR, "best.pt")
assert os.path.exists(best), "no best.pt yet (first one is written at the first evaluation)"
ck = torch.load(best, map_location="cpu", weights_only=False)
import arch3
m = arch3.DecoderLM(arch3.LMConfig(**ck["config"]["model"])); m.load_state_dict(ck["model"])
n = sum(p.numel() for p in m.parameters())
assert ck["config"]["module"] == "arch3" and ck["config"]["model"]["vocab_size"] == 32000
shutil.copy(best, os.path.join(exp, "best.pt"))
shutil.copy(TOKENIZER, os.path.join(WORK, "export", "legal_bpe.json"))
shutil.copy(os.path.join(DATA_DIR, "manifest.json"), os.path.join(WORK, "export", "pretrain_manifest.json"))
print(f"exported {n / 1e6:.1f}M params | step {ck['step']} | tokens {ck['tokens'] / 1e9:.2f}B | best val {ck['best_val']:.4f}")
print("folder:", os.path.join(WORK, "export"), "-> e.g. `kaggle datasets create -p export` (private), or download it")
print("In the Phase-2+ notebooks it is found like the others (pattern run_B3/best.pt); add it to PHASE1 as a new key.")